In [ ]:
import nltk
import numpy as np
from collections import Counter, defaultdict
import math
import random

nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
with open('/content/office_script_clean.txt', 'r') as f:
    text = f.read()

sample_corpus = [s.strip() for s in text.split('\n') if s.strip()]

def preprocess(sentences):
    tokenized_sentences = []
    for sentence in sentences:
        tokens = ['<s>'] + word_tokenize(sentence.lower()) + ['</s>']
        tokenized_sentences.append(tokens)
    return tokenized_sentences

In [ ]:
random.shuffle(sample_corpus)

train_split_idx = int(len(sample_corpus) * 0.8)
test_split_idx = int(len(sample_corpus) * 0.9)

train_corpus = sample_corpus[:train_split_idx]
test_corpus = sample_corpus[train_split_idx:test_split_idx]

tokenized_train_data = preprocess(train_corpus)
tokenized_test_data = preprocess(test_corpus)

print(f"Training corpus size: {len(train_corpus)}")
print(f"Testing corpus size: {len(test_corpus)}")
print(f"Sample tokenized training sentence: {tokenized_train_data[0]}")
print(f"Sample tokenized testing sentence: {tokenized_test_data[0]}")

Training corpus size: 43688
Testing corpus size: 5461
Sample tokenized training sentence: ['<s>', 'jim', ':', 'what', '?', '</s>']
Sample tokenized testing sentence: ['<s>', 'sweeney', 'todd', ':', 'you', "'re", 'the', 'guy', 'who', 'booed', 'me', '.', '</s>']


In [ ]:
class BigramModel:
    def __init__(self, tokenized_sentences):
        self.unigram_counts = Counter()
        self.bigram_counts = Counter()
        self.vocabulary = set()

        for sentence in tokenized_sentences:
            for i in range(len(sentence)):
                self.unigram_counts[sentence[i]] += 1
                self.vocabulary.add(sentence[i])
                if i > 0:
                    self.bigram_counts[(sentence[i-1], sentence[i])] += 1

        self.v_size = len(self.vocabulary)

    def get_probability(self, w_prev, w_curr):
        # Laplace (add-1) smoothing
        numerator = self.bigram_counts[(w_prev, w_curr)] + 1
        denominator = self.unigram_counts[w_prev] + self.v_size
        return numerator / denominator

model = BigramModel(tokenized_train_data)
print(f"Vocabulary size: {model.v_size}")

Vocabulary size: 18861


In [ ]:
def generate_sentence(model, max_len=20):
    sentence = ['<s>']
    while sentence[-1] != '</s>' and len(sentence) < max_len:
        prev_word = sentence[-1]

        # Sample next word based on probabilities
        words = list(model.vocabulary)
        probs = [model.get_probability(prev_word, w) for w in words]

        # Normalize probabilities to ensure they sum to 1 (due to floating point/sampling precision)
        probs = np.array(probs) / sum(probs)

        next_word = np.random.choice(words, p=probs)
        sentence.append(next_word)

    return ' '.join(sentence[1:-1])

print("Generated Sentences:")
for _ in range(5):
    print(f"- {generate_sentence(model)}")

Generated Sentences:
- pam heartwarming bestiality strengths acts mama rally slater swimming deathly so-i cancel usher psychiatrist program it's-in whoa-oh-oh gideon
- phil rockin services coaster mall rockefeller hearing coastal snob noon aha somersault catastrophe pate river extraordinaire announcer thermometer
- darryl milkshake uncool corvette usual lion forgiving cruiser sperm triplex jog burrito snagged newcomers underdressed revolutionary nuns ya
- katy sore eludes ram doing- target emergencies yes- interviewing stree sneaking mutato merci celebrated outdoors ro onion racy
- pam facilitator copperfield sorrys madhouse chemicals x-factor bubbly healthiest supplies rundown varicose cafeteria posters friday seconds friends- <s>


In [ ]:
def calculate_perplexity(model, test_sentences):
    # The preprocess function is already defined to tokenize sentences correctly
    tokenized_test = preprocess(test_sentences)
    log_probability = 0
    n = 0

    for sentence in tokenized_test:
        for i in range(1, len(sentence)):
            prob = model.get_probability(sentence[i-1], sentence[i])
            if prob == 0:
                prob = 1e-10
            log_probability += math.log2(prob)
            n += 1

    if n == 0: return float('inf') # Handle empty test set case

    avg_log_prob = log_probability / n
    perplexity = 2 ** (-avg_log_prob)
    return perplexity

ppl = calculate_perplexity(model, test_corpus)
print(f"Perplexity on test set (Laplace Smoothed): {ppl:.2f}")

Perplexity on test set (Laplace Smoothed): 357.74
